In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import seaborn as sns

In [ ]:
# Load the dataset
data_url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv'
print("===================Data URL==================")
print(data_url)
print()
print("===================DataFrame==================")
df = pd.read_csv(data_url)
print(df)

In [ ]:
# By default, it returns the first 5 rows. 
df.head()

In [ ]:
# Print summary
df.info()

In [ ]:
# Drop rows with missing values and select some features
df = df.dropna()
features = ['horsepower','weight','displacement','acceleration','cylinders']
target = 'mpg'

In [ ]:
# Print summary
df.info()
df.index

# `dropna()`, Index Behavior, and Memory Usage

## Why did only `horsepower` show 392 (not 398) before `dropna()`?

"Non-Null Count" = number of values in that column that are **not** `NaN`.

- All columns had 398 non-null values **except** `horsepower`, which had 392.
- That means **6 rows** had a missing value specifically in `horsepower`, while every other column was fully populated for all 398 rows.
- After `dropna()`, those 6 rows are removed entirely, so every column now shows 392 — there are no missing values left anywhere.

---

## Why "392 entries, 0 to 397"?

`dropna()` **deletes rows** but does **not renumber the index**.

- Original index: `RangeIndex` from 0 to 397 (398 labels, no gaps).
- After dropping the 6 rows with missing `horsepower`, the remaining rows **keep their original index labels**.
- So the labels still span 0 → 397, but 6 numbers in that range are simply missing (the ones that got dropped) → 392 actual rows, spread across a range of 398 possible labels.

Confirm with:
```python
df.index
# Index([0, 1, 2, 3, ..., 395, 396, 397], dtype='int64', length=392)
```

To reset to a clean 0-to-391 index:
```python
df = df.reset_index(drop=True)
```

---

## Why did memory usage go **up** after dropping rows?

This is about the **index**, not the data itself.

| Before | After |
|---|---|
| `RangeIndex` | `Index` (Int64Index) |

- A `RangeIndex` is virtually free to store — it's just `start`, `stop`, `step` (like Python's `range()`), regardless of how many rows it represents.
- Once rows are dropped **non-contiguously** (gaps in the middle), pandas can no longer represent the index as a simple range.
- It converts to a regular `Int64Index`, which must **explicitly store all 392 integers** in memory.

**Net effect:** the memory saved by removing 6 rows of data is smaller than the memory cost of switching from a "free" RangeIndex to an explicitly stored integer array → total memory usage goes **up** (28.1 KB → 30.6 KB).

---

## Key takeaway

 `dropna()` (and most row-filtering operations) preserve original index labels and can silently convert a cheap `RangeIndex` into a more expensive stored index.


In [ ]:
# Calculate correlation matrix
correlation_matrix = df[features+[target]].corr()

# Plot heatmap of feature importance
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap for Feature Importance')
plt.show

# `correlation_matrix = df[features + [target]].corr()`

This single line combines several separate ideas. Breaking it apart piece by piece:

---

## 1. `df` is just a variable name

```python
df = pd.read_csv(data_url)
```

- `df` is not special syntax — it's a **variable**, same as `x`, `data`, or `banana`. It could be named anything.
- It's the label pointing to the table of data loaded from the CSV.
- Every time `df` appears later in the code, it means: *"go get that same table I loaded earlier."*

---

## 2. `features` and `target` are just lists / strings of column names

```python
features = ['cylinders', 'displacement', 'weight', 'acceleration']
target = 'mpg'
```

- `features` → a **list** of column name strings.
- `target` → a single **string** (just one column name).

---

## 3. `features + [target]` — combining two lists

In Python, `+` between two lists means **concatenate** (join end-to-end), not math addition.

```python
features + [target]
# ['cylinders', 'displacement', 'weight', 'acceleration'] + ['mpg']
# → ['cylinders', 'displacement', 'weight', 'acceleration', 'mpg']
```

- `target` must be wrapped in brackets — `[target]` — to turn it into a one-item **list**, because you can only concatenate a list with another list (not a list with a plain string).
- Without the brackets (`features + target`), Python raises a `TypeError`.

**This step produces only a list of names — no actual data yet.** Think of it like a shopping list: just text, no groceries.

You can split this into its own step to see it clearly:
```python
selected_columns = features + [target]
print(selected_columns)
# → ['cylinders', 'displacement', 'weight', 'acceleration', 'mpg']
```

---

## 4. `df[...]` — indexing/selecting columns

Square brackets `[]` after a variable mean **"look inside this object and pull something out."** This is the same general pattern as:

```python
my_list[0]      # get item at position 0
my_string[1]    # get character at position 1
df[...]         # get column(s) from the table
```

What you get from `df[...]` depends on what's inside the brackets:

| Syntax | Input type | Returns |
|---|---|---|
| `df['mpg']` | single string | one column (Series, 1D) |
| `df[['mpg']]` | list with one name | one column (DataFrame, 2D) |
| `df[['mpg', 'weight']]` | list of names | multiple columns (DataFrame) |
| `df[features + [target]]` | list of names | multiple columns (DataFrame) |

So `df[features + [target]]` selects **only the chosen feature columns + the target column** out of `df` — leaving out anything irrelevant (like a `name` or `origin` column you don't want in the correlation check).

This is the step where "just column names" becomes **real numeric data**:

```python
selected_columns              # ['cylinders', ..., 'mpg']         ← just names, no data
df[selected_columns]          # actual table of real values       ← real data pulled from df
```

**Analogy:** `selected_columns` is the shopping list. `df` is the grocery store. `df[selected_columns]` is walking in and grabbing exactly those items off the shelf.

---

## 5. `.corr()` — computing the correlation matrix

```python
df[features + [target]].corr()
```

- `.corr()` computes the **pairwise correlation coefficient** between every pair of numeric columns in the DataFrame it's called on.
- It needs **actual numeric data** to work — this is why it's called on `df[...]` (real values), not on `selected_columns` (just a list of names, which has no `.corr()` method at all and would raise `AttributeError`).
- The result is a square matrix: each row/column is one of your selected features (+ target), and each cell shows how strongly that pair of variables correlates (from -1 to 1).

---

## Putting it all together

```python
correlation_matrix = df[features + [target]].corr()
```

Reads as:

> "From `df`, select only the feature columns plus the target column, then compute the correlation matrix on that selection."

Equivalent, spelled out in separate steps:

```python
selected_columns = features + [target]        # build the combined list of names
subset_df = df[selected_columns]               # pull those columns' real data out of df
correlation_matrix = subset_df.corr()           # compute correlations on that real data
```

---

## Why not just use the full `df`?

```python
df.corr()   # would try to correlate ALL columns, including non-numeric/irrelevant ones
```

- `df` often has extra columns (IDs, names, categories) that aren't useful or even numeric for correlation analysis.
- Selecting `features + [target]` first keeps the correlation matrix focused only on the variables you actually care about.

---

## Key takeaway

> `df[features + [target]].corr()` = **build a list of the column names you want → use that list to pull real data out of the DataFrame → compute correlations on that real data.**


In [ ]:
print("===================features & [features]==================")
print(features)
print([features])
print()

print("===================target & [target]==================")
print(target)
print([target])
print()

print("===================[features+[target]]==================")
print([features+[target]])
print()

print("===================df[features+[target]]e==================")
print(df[features+[target]])
print()

In [ ]:
print("===================correlation_matrix==================")
print(correlation_matrix)

In [ ]:
# Select features from the correlation matrix and use mpg as the target variable
X = df[['horsepower', 'weight', 'displacement', 'cylinders']].values
y = df['mpg'].values

In [ ]:
print(X)
print('=====================================')
print(y)

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

`train_test_split()` and Multiple Variable Assignment

## 1. What `X` and `y` are

```python
X = df[['horsepower', 'weight', 'displacement', 'cylinders']].values
y = df['mpg'].values
```

- `X` → the **features** (inputs) — horsepower, weight, displacement, cylinders — for all rows.
- `y` → the **target** (output) — `mpg` — for all rows.
- `.values` converts the pandas DataFrame/Series into a plain NumPy array, which is the format most ML models expect.

---

## 2. `X_train, X_test, y_train, y_test = ...` — Tuple Unpacking

This is a Python feature called **tuple unpacking**: when a function returns multiple values packed together, you can assign each one to its own variable in a single line — as long as the number of variables on the left matches the number of values returned on the right.

```python
a, b, c = (1, 2, 3)
print(a)  # 1
print(b)  # 2
print(c)  # 3
```

`train_test_split()` returns **4 values** as a tuple, so **4 variable names** are needed to catch them:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

Equivalent to writing it manually:
```python
result = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = result[0]
X_test  = result[1]
y_train = result[2]
y_test  = result[3]
```

⚠️ **Order matters** — `train_test_split` always returns values in this fixed order: `X_train, X_test, y_train, y_test`.

---

## 3. What `train_test_split()` actually does

It **randomly divides** `X` and `y` into two separate groups:

| Set | Purpose | Size (with `test_size=0.2`) |
|---|---|---|
| **Training set** (`X_train`, `y_train`) | Used to *train* the model — it learns patterns from this data | 80% of rows |
| **Test set** (`X_test`, `y_test`) | Used to *evaluate* the model — data it has never seen | 20% of rows |

**Why split at all?**
If a model is trained and tested on the exact same data, it can just "memorize" it and look artificially perfect. Testing on unseen data reveals how well the model actually **generalizes** to new, real-world cases it hasn't encountered before.

`X_train`/`y_train` and `X_test`/`y_test` stay aligned row-for-row — whichever rows go into `X_train` also send their matching `mpg` values into `y_train`, and the same pairing applies to the test set.

---

## 4. Why `random_state=42`?

This isn't about generating fake data — it's about controlling **how the random split happens**.

- `train_test_split` shuffles rows randomly before splitting them, so the split isn't biased (e.g., all high-mpg cars accidentally ending up in one set).
- "Random" in code isn't truly random — it's generated by an algorithm that needs a starting point, called a **seed**.
- `random_state=42` fixes that seed to a specific number.

**Effect:**
- With `random_state=42` → every re-run of the code produces the **exact same split** (same rows in `X_train`/`X_test` every time).
- Without setting `random_state` (or `None`) → a **different random split every run**, making results impossible to reproduce or debug reliably.

> The number `42` itself is arbitrary (a nod to *The Hitchhiker's Guide to the Galaxy*) — any integer works the same way. What matters is that it's **fixed**, not what the specific number is.

---

## Key takeaway

> `train_test_split()` returns 4 values at once (train/test splits for both features and target), which is why 4 variables are assigned on the left in one line. `random_state` makes the "random" split **reproducible**, so the same train/test division happens every time the code runs — keeping results consistent and comparable across runs.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("This is X1: ", X_test[:5])   # first 5 rows that landed in the test set

# Run it again with the same random_state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("This is X2: ",X_test[:5])   # identical output — same rows, same order

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
print("This is X3: ",X_test[:5])   # different rows than before

# `random_state` — Concrete Example

## The confusion

> "`random_state=42` doesn't mean row 5, 7, 9 are always test rows no matter what." vs. "I set 80% training / 20% test — so how does that work?"

The short answer: **as long as your data, `test_size`, and `random_state` all stay the same between runs, you WILL get the exact same rows in train/test every time.** The split only changes if the *inputs* to `train_test_split()` change.

---

## Concrete example (tiny dataset, 10 rows: 0–9)

With `test_size=0.2` → 80% train (8 rows) / 20% test (2 rows).

### Run 1 — `random_state=42`
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_test ends up containing rows: [2, 7]
```

### Run 2 — same code, same data, same `random_state=42`
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_test ends up containing rows: [2, 7]   ← identical, every time
```

✅ **Yes — as long as data + `test_size` + `random_state` are unchanged, you always get rows `[2, 7]` in the test set.** That's what "reproducible" means. Your 80/20 split is locked in.

---

## What actually breaks the "same rows every time" guarantee

If the **data itself** changes — e.g. a row gets dropped, added, or reordered — and you run the *same* `random_state=42` again:

```python
# Same random_state=42, but X/y is now different/reordered
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_test might now contain rows: [4, 9]   ← different rows than before!
```

Same seed number (42), but a **different result** — because the seed isn't a fixed lookup table saying "row 2 and row 7 are always test." It's a fixed **shuffle recipe** — and recipe #42 applied to *different* input data naturally produces a different-looking split.

---

## The rule, in one sentence

> `random_state=42` guarantees: **same data in → same split out, every time.**
> It does **not** guarantee: rows 2 and 7 are always the test set, regardless of what data is fed in.

---

## Practical takeaway for a notebook where data doesn't change

If you're just re-running the same cell without altering `X`/`y`:

| Stays the same | Result |
|---|---|
| Data (`X`, `y`) ✅ unchanged | |
| `test_size=0.2` ✅ unchanged | Same rows in **train** every run |
| `random_state=42` ✅ unchanged | Same rows in **test** every run |

→ **Your 80/20 split is effectively frozen** — identical rows in train and test, every single time you run the cell.

It only changes if you:
- Change the data (drop/add/reorder rows), **or**
- Change `test_size`, **or**
- Change `random_state` to a different number


In [ ]:
# Create a Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

In [ ]:
print("===================y_pred==================")
print(y_pred)
print("===================y_test==================")
print(y_test)

In [ ]:
# Evaluate the mode
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")

In [ ]:
# Plot the true vs predicted mpg
plt.figure(figsize=(16, 12))

plt.scatter(y_test, y_pred, alpha=0.75, color='blue', label='Predicted vs True')

plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--', label='Ideal Prediction Line')

# Calculate and plot prediction errors
for (true_value, predicted_value) in zip(y_test, y_pred):
    plt.plot([true_value, true_value], [true_value, predicted_value], color='gray', linestyle='-', linewidth=0.5)


plt.xlabel("True MPG")
plt.ylabel("Predicted MPG")
plt.title("True MPG vs Predicted MPG with Prediction Errors")
plt.legend()
plt.grid(True)
plt.xticks(np.arange(0, 45, 1))  
plt.yticks(np.arange(0, 45, 1))   
plt.show()
df.info()